# SPARK SQL

- 데이터를 처리하고 분석하기 위한 SQL 인터페이스 및 엔진 + 데이터 및 메타데이터를 관리
- 즉, SQL 쿼리로 Spark DataFrame을 다룰 수 있도록 하는 기능

- 관련 내용 정리
  - [Spark SQL과 Spark Catalog를 통한 테이블 관리 및 데이터 프로세싱](https://velog.io/@newnew_daddy/SPARK12)

## 1. 임시 테이블

In [1]:
!pip3 install pyspark


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488491 sha256=5547447368f6e766ffb1e3f0a67c42c6d1b0b1624709de5781e2f67e4756e0fc
  Stored in directory: /root/.cache/pip/wheels/80/1d/60/2c256ed38dddce2fdd93be545214a63e02fbd8d74fb0b7f3a6
Successfully built pyspark


In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .master("local") \
        .appName("Colab") \
        .getOrCreate()

In [2]:
# in-memory -> 휘발성 테이블(런타임 종료시 등록된 테이블도 삭제) default
#hive -> 영구 테이블(런타임 종료되어도 등록된 테이블 존재)

spark.conf.get("spark.sql.catalogImplementation")

'in-memory'

#### 1) 테이블 생성

In [3]:
study = spark.read.parquet("./study_his.parquet")
study.show(3)

+-----+-------+--------+-----------+
|  idx|proc_ym|proc_ymd|    pointnm|
+-----+-------+--------+-----------+
|88311| 202306|20230628|한글 스피치|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
+-----+-------+--------+-----------+
only showing top 3 rows



In [4]:
# _query = "SHOW DATABASES"
_query = "SHOW TABLES FROM default"

spark.sql(_query).show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



In [5]:
#데이터베이스 생성
spark.sql("CREATE DATABASE temp").show()

++
||
++
++



In [6]:
spark.sql("CREATE DATABASE temp2").show()

++
||
++
++



In [7]:
spark.catalog.listDatabases() #이렇게 정보 볼 수도 있음

[Database(name='default', catalog='spark_catalog', description='default database', locationUri='file:/content/spark-warehouse'),
 Database(name='temp', catalog='spark_catalog', description='', locationUri='file:/content/spark-warehouse/temp.db'),
 Database(name='temp2', catalog='spark_catalog', description='', locationUri='file:/content/spark-warehouse/temp2.db')]

In [14]:
# study 테이블 등록
study.createOrReplaceTempView("study")
study.createOrReplaceTempView("study2")
# study.createOrReplaceTempView("temp.study2") #->이런식으로 db지정해서 등록 할 수도 있음.

In [15]:
_query = "SHOW TABLES FROM default"

spark.sql(_query).show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|         |    study|       true|
|         |   study2|       true|
+---------+---------+-----------+



In [16]:
spark.catalog.listTables("Default")

[Table(name='study', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='study2', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [17]:
spark.catalog.listColumns("study")

[Column(name='idx', description=None, dataType='string', nullable=True, isPartition=False, isBucket=False),
 Column(name='proc_ym', description=None, dataType='string', nullable=True, isPartition=False, isBucket=False),
 Column(name='proc_ymd', description=None, dataType='string', nullable=True, isPartition=False, isBucket=False),
 Column(name='pointnm', description=None, dataType='string', nullable=True, isPartition=False, isBucket=False)]

#### 2) 저장된 테이블 조회

In [18]:
# 1) sql 쿼리로 읽는 방법
spark.sql("SELECT * FROM study LIMIT 10").show()

+-----+-------+--------+-----------+
|  idx|proc_ym|proc_ymd|    pointnm|
+-----+-------+--------+-----------+
|88311| 202306|20230628|한글 스피치|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
+-----+-------+--------+-----------+



In [19]:
spark.sql("SELECT proc_ym, COUNT(proc_ym) FROM study GROUP BY proc_ym ORDER BY proc_ym ASC").show()

+-------+--------------+
|proc_ym|count(proc_ym)|
+-------+--------------+
| 202304|         44422|
| 202305|         22713|
| 202306|         24994|
+-------+--------------+



In [20]:
# 2) read table 함수를 통해 읽는 방법
spark.read.table("study").show()

+-----+-------+--------+-----------+
|  idx|proc_ym|proc_ymd|    pointnm|
+-----+-------+--------+-----------+
|88311| 202306|20230628|한글 스피치|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
| 8604| 202306|20230606| 중학 3학년|
|89542| 202305|20230509| 중학 3학년|
|23940| 202305|20230516| 중학 1학년|
|23940| 202305|20230516| 중학 1학년|
|40502| 202304|20230412| 중학 1학년|
| 1741| 202304|20230405| 중학 1학년|
|50161| 202304|20230428| 중학 1학년|
+-----+-------+--------+-----------+
only showing top 20 rows



## 2. 영구 테이블

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .master("local") \
        .appName("Colab") \
        .enableHiveSupport() \
        .getOrCreate()

In [2]:
spark.conf.get("spark.sql.catalogImplementation")

'hive'

#### 1) Database 생성 및 조회

In [3]:
spark.sql("CREATE DATABASE kdt7")

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/usr/local/lib/python3.10/dist-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [4]:
spark.sql("SHOW DATABASES").show()

+---------+
|namespace|
+---------+
|  default|
|     kdt7|
+---------+



In [5]:
spark.catalog.listDatabases()

[Database(name='default', catalog='spark_catalog', description='Default Hive database', locationUri='file:/content/spark-warehouse'),
 Database(name='kdt7', catalog='spark_catalog', description='', locationUri='file:/content/spark-warehouse/kdt7.db')]

#### 2) Table 생성 및 조회

In [9]:
# CREATE TABLE 명령을 통한 생성과 등록

# create_query="CREATE TABLE first(id int, name String)"
create_query="CREATE TABLE kdt7.second(id int, name String)" # 데이터 베이스 지정하고 그 데이터베이스에 테이블 생성
spark.sql(create_query)

DataFrame[]

In [6]:
# spark.sql("SHOW TABLES FROM default").show()
spark.sql("SHOW TABLES FROM kdt7").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|     kdt7|   second|      false|
|     kdt7|    study|      false|
+---------+---------+-----------+



In [14]:
#INSERT QUERY 명령
# insert_query = "INSERT INTO first VALUES(1,'Tom'),(2,'Ann')"
insert_query = "INSERT INTO kdt7.second VALUES(1,'Paul'),(2,'Marie')"

spark.sql(insert_query)

DataFrame[]

In [16]:
# spark.sql("SELECT * FROM first").show()
spark.sql("SELECT * FROM kdt7.second").show()

+---+-----+
| id| name|
+---+-----+
|  1| Paul|
|  2|Marie|
+---+-----+



In [17]:
# write 명령을 통한 테이블 등록

study=spark.read.parquet("./study_his.parquet")

In [20]:
# study.write.saveAsTable("study")
study.write.saveAsTable("kdt7.study")

In [23]:
spark.sql("SHOW TABLES FROM default").show()
# spark.sql("SHOW TABLES FROM kdt7").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|  default|    first|      false|
|  default|    study|      false|
|         |  study_1|       true|
+---------+---------+-----------+



In [22]:
#createOrReplaceTempView 명령 사용
study.createOrReplaceTempView("study_1")

In [ ]:
# 1. create table -> 생성된 테이블과 데이터가 영구 저장.
#2. spark.write.saveAsTable -> 생성된 테이블과 데이터가 영구 저장
# 3. createOrReplaveTempView ->생성된 테이블과 데이터가 일시 저장(휘발성)

#### 3) 저장된 테이블 조회

In [7]:
from IPython.display import clear_output

def execute_spark_queries():
    while True:
        _query = input("Enter your Spark SQL query (type 'q' to quit): ")
        clear_output(wait=True)
        if _query.lower() == "q":
            break
        try:
            result = spark.sql(_query)
            result.show()
        except Exception as e:
            print(f"Error executing query: {e}")

execute_spark_queries()

Error executing query: 
[PARSE_SYNTAX_ERROR] Syntax error at or near 'FROM': missing 'FUNCTIONS'.(line 1, pos 13)

== SQL ==
SHOW DATABAS FROM default
-------------^^^

Enter your Spark SQL query (type 'q' to quit): q
